In [ ]:
import os
import json
from tqdm import tqdm
from llama_cpp import Llama

# Configuration
MODEL_PATH = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/new_gemma4_12B_q8.gguf"
INPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/TiJian94.json"
OUTPUT_JSON = "/data/projects/mayapeng/hosptials_model/merge_model/merge_q8/result_94/results_with_gemma4.json"

GPU_LAYERS = -1
N_CTX = 32768               
N_BATCH = 1024
N_UBATCH = 512
N_THREADS = 4
MAIN_GPU = 0
MAX_TOKENS = 4096
TEMPERATURE = 0.1
TOP_P = 0.8
SAVE_EVERY = 10

# Load model
print("Loading Gemma4 GGUF...")
llm = Llama(
    model_path=MODEL_PATH, n_gpu_layers=GPU_LAYERS,
    n_ctx=N_CTX, n_batch=N_BATCH, n_ubatch=N_UBATCH,
    n_threads=N_THREADS, verbose=False
)
print("Model Loaded!")

# Read data
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    dataset = json.load(f)
print(f"Total records: {len(dataset)}")

# Resume from checkpoint
results = []
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, "r", encoding="utf-8") as f:
        results = json.load(f)
    start_idx = len(results)
    print(f"Resuming from record {start_idx}")
else:
    start_idx = 0

# Inference
for idx in tqdm(range(start_idx, len(dataset)), desc="Inference"):
    item = dataset[idx]
    instruction = item.get("instruction", "")
    input_text = item.get("input", "")
    gt_output = item.get("output", "")

    try:
        messages = [{"role": "user", "content": instruction + "\n" + input_text}]
        response = llm.create_chat_completion(
            messages=messages, temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS
        )
        answer = response["choices"][0]["message"]["content"].strip()
        results.append({
            "instruction": instruction, "input": input_text, "output": gt_output, "output_gemma4": answer
        })
    except Exception as e:
        print(f"\nFailed on record {idx}: {e}")
        results.append({
            "instruction": instruction, "input": input_text, "output": gt_output, "output_gemma4": "", "error": str(e)
        })

    # Save every 10 records
    if len(results) % SAVE_EVERY == 0:
        with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
        print(f"Saved {len(results)} records")

# Final save
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"All inference completed! Results saved to: {OUTPUT_JSON}")